In [6]:
import numpy as np
import cv2
import os
import copy
import scipy.optimize as optimize
import matplotlib.pyplot as plt

# -----------------------------
# Utility Functions
# -----------------------------

def read_images(folder="distored_calib_imgs"):
    """Read all images from folder."""
    if not os.path.exists(folder):
        raise FileNotFoundError(f"{folder} does not exist")
    
    image_list = sorted([os.path.join(folder, f) 
                         for f in os.listdir(folder) if f.lower().endswith(".jpg")])
    
    images = [cv2.imread(f) for f in image_list]
    print("Loaded", len(images), "images")
    return images

def draw_corners(img, corners, color=(0,0,255)):
    """Draw corners on image."""
    img_copy = img.copy()
    for c in corners:
        cv2.circle(img_copy, (int(c[0]), int(c[1])), 7, color, -1)
    return img_copy

def plot_corners(img, corners, name):
    """Save image with corners drawn."""
    img_with_corners = draw_corners(img, corners)
    os.makedirs("Results", exist_ok=True)
    cv2.imwrite(f"Results/{name}.png", img_with_corners)

# -----------------------------
# Chessboard & World Coordinates
# -----------------------------

def find_img_corners(images, checker_size=(9,6)):
    """Detect chessboard corners in all images."""
    all_corners = []
    for idx, img in enumerate(images):
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        found, corners = cv2.findChessboardCorners(gray, checker_size)
        if not found:
            raise RuntimeError(f"Chessboard not found in image {idx}")
        corners = corners.squeeze(1).astype(np.float32)
        plot_corners(img, corners, f"original_corners_{idx+1}")
        all_corners.append(corners)
    return np.array(all_corners)  # shape (N_images, N_points, 2)

def find_world_corners(checker_size=(9,6), square_size=21.5):
    """Generate world coordinates for checkerboard points."""
    corners = []
    for i in range(checker_size[1]):
        for j in range(checker_size[0]):
            corners.append([j*square_size, i*square_size])
    return np.array(corners, dtype=np.float32)  # shape (N_points, 2)

# -----------------------------
# Homography & Intrinsics
# -----------------------------

def find_homography(img_corners, world_corners):
    """Compute homography H from world to image points."""
    N = img_corners.shape[0]
    A = []
    for i in range(N):
        X, Y = world_corners[i]
        u, v = img_corners[i]
        A.append([-X, -Y, -1, 0, 0, 0, u*X, u*Y, u])
        A.append([0, 0, 0, -X, -Y, -1, v*X, v*Y, v])
    A = np.array(A)
    U, S, Vh = np.linalg.svd(A)
    H = Vh[-1].reshape(3,3)
    H /= H[2,2]
    return H

def all_homographies(images_corners, world_corners):
    """Compute homography for all images."""
    return np.array([find_homography(img_c, world_corners) 
                     for img_c in images_corners])

def V_ij(H, i, j):
    """Compute Vij vector for Zhang calibration."""
    return np.array([
        H[0,i]*H[0,j],
        H[0,i]*H[1,j] + H[1,i]*H[0,j],
        H[1,i]*H[1,j],
        H[2,i]*H[0,j] + H[0,i]*H[2,j],
        H[2,i]*H[1,j] + H[1,i]*H[2,j],
        H[2,i]*H[2,j]
    ])

def compute_B(H_all):
    """Compute B matrix from homographies."""
    V = []
    for H in H_all:
        V.append(V_ij(H,0,1))
        V.append(V_ij(H,0,0) - V_ij(H,1,1))
    V = np.array(V)
    _, _, Vh = np.linalg.svd(V)
    b = Vh[-1]
    B = np.array([[b[0], b[1], b[3]],
                  [b[1], b[2], b[4]],
                  [b[3], b[4], b[5]]])
    return B

def compute_K(B):
    """Compute camera intrinsic matrix K from B."""
    v0 = (B[0,1]*B[0,2] - B[0,0]*B[1,2]) / (B[0,0]*B[1,1]-B[0,1]**2)
    lmbda = B[2,2] - (B[0,2]**2 + v0*(B[0,1]*B[0,2]-B[0,0]*B[1,2]))/B[0,0]
    alpha = np.sqrt(lmbda / B[0,0])
    beta = np.sqrt(lmbda*B[0,0] / (B[0,0]*B[1,1]-B[0,1]**2))
    gamma = -B[0,1]*alpha**2*beta/lmbda
    u0 = gamma*v0/beta - B[0,2]*alpha**2/lmbda
    K = np.array([[alpha, gamma, u0],
                  [0, beta, v0],
                  [0, 0, 1]])
    return K

# -----------------------------
# Extrinsics
# -----------------------------

def extrinsics(K, H_all):
    """Compute rotation and translation matrices from homography."""
    Rt_all = []
    K_inv = np.linalg.pinv(K)
    for H in H_all:
        h1, h2, h3 = H[:,0], H[:,1], H[:,2]
        lam = 1.0 / np.linalg.norm(K_inv @ h1)
        r1 = lam * K_inv @ h1
        r2 = lam * K_inv @ h2
        r3 = np.cross(r1, r2)
        t = lam * K_inv @ h3
        Rt = np.column_stack((r1,r2,r3,t))
        Rt_all.append(Rt)
    return Rt_all

# -----------------------------
# Distortion & Reprojection
# -----------------------------

def get_A_matrix(param):
    """Convert 7-element vector to K and distortion."""
    alpha, gamma, beta, u0, v0, k1, k2 = param
    K = np.array([[alpha, gamma, u0],
                  [0, beta, v0],
                  [0, 0, 1]])
    k_dist = np.array([float(k1), float(k2)])  # Ensure scalars
    return K, k_dist

def get_parameters(K, k_dist):
    """Convert K and k_dist to 7-element vector."""
    return np.array([K[0,0], K[0,1], K[1,1], K[0,2], K[1,2], k_dist[0], k_dist[1]])

def calc_points_image(K, k_dist, Rt, world_points):
    """Project world points to image with distortion."""
    points_proj = []
    k1, k2 = float(k_dist[0]), float(k_dist[1])
    u0, v0 = K[0,2], K[1,2]
    alpha, beta = K[0,0], K[1,1]
    
    for X, Y in world_points:
        world_h = np.array([X,Y,0,1])
        img_h = Rt @ world_h
        u, v = img_h[0]/img_h[2], img_h[1]/img_h[2]
        xn = (u - u0)/alpha
        yn = (v - v0)/beta
        r2 = xn**2 + yn**2
        xnd = xn*(1 + k1*r2 + k2*r2**2)
        ynd = yn*(1 + k1*r2 + k2*r2**2)
        uc = xnd*alpha + u0
        vc = ynd*beta + v0
        points_proj.append([uc, vc])
    return np.array(points_proj)

def rms_error_reprojection(K, k_dist, Rt_all, images_corners, world_corners):
    """Compute mean reprojection error over all images."""
    error_all = []
    reprojected_all = []
    for img_pts, Rt in zip(images_corners, Rt_all):
        proj_pts = calc_points_image(K, k_dist, Rt, world_corners)
        reprojected_all.append(proj_pts)
        error = np.linalg.norm(proj_pts - img_pts, axis=1)
        error_all.append(np.mean(error))
    return np.array(error_all), reprojected_all

def reprojection_loss(param, Rt_all, images_corners, world_corners):
    """Loss function for least_squares."""
    K, k_dist = get_A_matrix(param)
    errors, _ = rms_error_reprojection(K, k_dist, Rt_all, images_corners, world_corners)
    return errors

# -----------------------------
# Main Calibration Pipeline
# -----------------------------

def main():
    images = read_images()
    checker_size = (9,6)
    square_size = 21.5
    
    images_corners = find_img_corners(images, checker_size)
    world_corners = find_world_corners(checker_size, square_size)
    
    H_all = all_homographies(images_corners, world_corners)
    B = compute_B(H_all)
    K = compute_K(B)
    Rt_all = extrinsics(K, H_all)
    k_dist = np.array([0.0,0.0])
    
    param_init = get_parameters(K, k_dist)
    
    # Optimize distortion parameters
    result = optimize.least_squares(reprojection_loss, param_init, method='lm',
                                    args=(Rt_all, images_corners, world_corners))
    K_new, k_new = get_A_matrix(result.x)
    
    print("Optimized K:\n", K_new)
    print("Optimized distortion [k1,k2]:", k_new)
    
    # Compute final reprojection error
    errors, reprojected_pts = rms_error_reprojection(K_new, k_new, Rt_all, images_corners, world_corners)
    print("Mean reprojection error (pixels):", np.mean(errors))
    
    # Save images with reprojected corners
    for idx, img in enumerate(images):
        img_proj = draw_corners(img, reprojected_pts[idx], color=(0,255,0))
        os.makedirs("Results", exist_ok=True)
        cv2.imwrite(f"Results/reprojected_{idx+1}.png", img_proj)

if __name__ == "__main__":
    main()


Loaded 13 images
Optimized K:
 [[-3.40374526e+10  4.13070855e+01  1.09308919e+03]
 [ 0.00000000e+00  2.03463864e+03  1.69419275e+03]
 [ 0.00000000e+00  0.00000000e+00  1.00000000e+00]]
Optimized distortion [k1,k2]: [-5.61372688  6.6302077 ]
Mean reprojection error (pixels): 348.75899987004044
